# HEMA FAIR OMOP ETL
Creates a PostgreSQL OMOP CDM database, imports vocabulary files, and maps HEMA source data to OMOP tables.

## Connection Settings

In [ ]:
DB_HOST     = "51.15.26.149"
DB_PORT     = 5432
DB_NAME     = "trainee_01"   # replace with your trainee number
DB_USER     = "trainee_01"   # replace with your trainee number
DB_PASSWORD = ""             # fill in your password

print(f"Connecting to {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Setup

In [ ]:
# !pip install psycopg2-binary sqlalchemy pandas

import psycopg2
import pandas as pd
from sqlalchemy import create_engine

con = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
con.autocommit = True

# SQLAlchemy engine — used for pandas to_sql (CSV bulk loading)
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def execute(sql):
    with con.cursor() as cur:
        cur.execute(sql)

def query(sql):
    return pd.read_sql_query(sql, con)

print("Connected to PostgreSQL.")

## Create Schemas

In [ ]:
execute("CREATE SCHEMA IF NOT EXISTS omop")
execute("CREATE SCHEMA IF NOT EXISTS results")

query("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name
""")

## Create OMOP CDM Tables

In [ ]:
execute("""
CREATE TABLE omop.person (
    person_id integer NOT NULL,
    gender_concept_id integer NOT NULL,
    year_of_birth integer NOT NULL,
    month_of_birth integer NULL,
    day_of_birth integer NULL,
    birth_datetime TIMESTAMP NULL,
    race_concept_id integer NOT NULL,
    ethnicity_concept_id integer NOT NULL,
    location_id integer NULL,
    provider_id integer NULL,
    care_site_id integer NULL,
    person_source_value varchar(50) NULL,
    gender_source_value varchar(50) NULL,
    gender_source_concept_id integer NULL,
    race_source_value varchar(50) NULL,
    race_source_concept_id integer NULL,
    ethnicity_source_value varchar(50) NULL,
    ethnicity_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.observation_period (
    observation_period_id integer NOT NULL,
    person_id integer NOT NULL,
    observation_period_start_date date NOT NULL,
    observation_period_end_date date NOT NULL,
    period_type_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.visit_occurrence (
    visit_occurrence_id integer NOT NULL,
    person_id integer NOT NULL,
    visit_concept_id integer NOT NULL,
    visit_start_date date NOT NULL,
    visit_start_datetime TIMESTAMP NULL,
    visit_end_date date NOT NULL,
    visit_end_datetime TIMESTAMP NULL,
    visit_type_concept_id integer NOT NULL,
    provider_id integer NULL,
    care_site_id integer NULL,
    visit_source_value varchar(50) NULL,
    visit_source_concept_id integer NULL,
    admitted_from_concept_id integer NULL,
    admitted_from_source_value varchar(50) NULL,
    discharged_to_concept_id integer NULL,
    discharged_to_source_value varchar(50) NULL,
    preceding_visit_occurrence_id integer NULL
);
""")

execute("""
CREATE TABLE omop.visit_detail (
    visit_detail_id integer NOT NULL,
    person_id integer NOT NULL,
    visit_detail_concept_id integer NOT NULL,
    visit_detail_start_date date NOT NULL,
    visit_detail_start_datetime TIMESTAMP NULL,
    visit_detail_end_date date NOT NULL,
    visit_detail_end_datetime TIMESTAMP NULL,
    visit_detail_type_concept_id integer NOT NULL,
    provider_id integer NULL,
    care_site_id integer NULL,
    visit_detail_source_value varchar(50) NULL,
    visit_detail_source_concept_id integer NULL,
    admitted_from_concept_id integer NULL,
    admitted_from_source_value varchar(50) NULL,
    discharged_to_source_value varchar(50) NULL,
    discharged_to_concept_id integer NULL,
    preceding_visit_detail_id integer NULL,
    parent_visit_detail_id integer NULL,
    visit_occurrence_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.condition_occurrence (
    condition_occurrence_id integer NOT NULL,
    person_id integer NOT NULL,
    condition_concept_id integer NOT NULL,
    condition_start_date date NOT NULL,
    condition_start_datetime TIMESTAMP NULL,
    condition_end_date date NULL,
    condition_end_datetime TIMESTAMP NULL,
    condition_type_concept_id integer NOT NULL,
    condition_status_concept_id integer NULL,
    stop_reason varchar(20) NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    condition_source_value varchar(50) NULL,
    condition_source_concept_id integer NULL,
    condition_status_source_value varchar(50) NULL
);
""")

execute("""
CREATE TABLE omop.drug_exposure (
    drug_exposure_id integer NOT NULL,
    person_id integer NOT NULL,
    drug_concept_id integer NOT NULL,
    drug_exposure_start_date date NOT NULL,
    drug_exposure_start_datetime TIMESTAMP NULL,
    drug_exposure_end_date date NOT NULL,
    drug_exposure_end_datetime TIMESTAMP NULL,
    verbatim_end_date date NULL,
    drug_type_concept_id integer NOT NULL,
    stop_reason varchar(20) NULL,
    refills integer NULL,
    quantity NUMERIC NULL,
    days_supply integer NULL,
    sig TEXT NULL,
    route_concept_id integer NULL,
    lot_number varchar(50) NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    drug_source_value varchar(50) NULL,
    drug_source_concept_id integer NULL,
    route_source_value varchar(50) NULL,
    dose_unit_source_value varchar(50) NULL
);
""")

execute("""
CREATE TABLE omop.procedure_occurrence (
    procedure_occurrence_id integer NOT NULL,
    person_id integer NOT NULL,
    procedure_concept_id integer NOT NULL,
    procedure_date date NOT NULL,
    procedure_datetime TIMESTAMP NULL,
    procedure_end_date date NULL,
    procedure_end_datetime TIMESTAMP NULL,
    procedure_type_concept_id integer NOT NULL,
    modifier_concept_id integer NULL,
    quantity integer NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    procedure_source_value varchar(50) NULL,
    procedure_source_concept_id integer NULL,
    modifier_source_value varchar(50) NULL
);
""")

execute("""
CREATE TABLE omop.device_exposure (
    device_exposure_id integer NOT NULL,
    person_id integer NOT NULL,
    device_concept_id integer NOT NULL,
    device_exposure_start_date date NOT NULL,
    device_exposure_start_datetime TIMESTAMP NULL,
    device_exposure_end_date date NULL,
    device_exposure_end_datetime TIMESTAMP NULL,
    device_type_concept_id integer NOT NULL,
    unique_device_id varchar(255) NULL,
    production_id varchar(255) NULL,
    quantity integer NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    device_source_value varchar(50) NULL,
    device_source_concept_id integer NULL,
    unit_concept_id integer NULL,
    unit_source_value varchar(50) NULL,
    unit_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.measurement (
    measurement_id integer NOT NULL,
    person_id integer NOT NULL,
    measurement_concept_id integer NOT NULL,
    measurement_date date NOT NULL,
    measurement_datetime TIMESTAMP NULL,
    measurement_time varchar(10) NULL,
    measurement_type_concept_id integer NOT NULL,
    operator_concept_id integer NULL,
    value_as_number NUMERIC NULL,
    value_as_concept_id integer NULL,
    unit_concept_id integer NULL,
    range_low NUMERIC NULL,
    range_high NUMERIC NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    measurement_source_value varchar(50) NULL,
    measurement_source_concept_id integer NULL,
    unit_source_value varchar(50) NULL,
    unit_source_concept_id integer NULL,
    value_source_value varchar(50) NULL,
    measurement_event_id bigint NULL,
    meas_event_field_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.observation (
    observation_id integer NOT NULL,
    person_id integer NOT NULL,
    observation_concept_id integer NOT NULL,
    observation_date date NOT NULL,
    observation_datetime TIMESTAMP NULL,
    observation_type_concept_id integer NOT NULL,
    value_as_number NUMERIC NULL,
    value_as_string varchar(60) NULL,
    value_as_concept_id integer NULL,
    qualifier_concept_id integer NULL,
    unit_concept_id integer NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    observation_source_value varchar(50) NULL,
    observation_source_concept_id integer NULL,
    unit_source_value varchar(50) NULL,
    qualifier_source_value varchar(50) NULL,
    value_source_value varchar(50) NULL,
    observation_event_id bigint NULL,
    obs_event_field_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.death (
    person_id integer NOT NULL,
    death_date date NOT NULL,
    death_datetime TIMESTAMP NULL,
    death_type_concept_id integer NULL,
    cause_concept_id integer NULL,
    cause_source_value varchar(50) NULL,
    cause_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.note (
    note_id integer NOT NULL,
    person_id integer NOT NULL,
    note_date date NOT NULL,
    note_datetime TIMESTAMP NULL,
    note_type_concept_id integer NOT NULL,
    note_class_concept_id integer NOT NULL,
    note_title varchar(250) NULL,
    note_text TEXT NOT NULL,
    encoding_concept_id integer NOT NULL,
    language_concept_id integer NOT NULL,
    provider_id integer NULL,
    visit_occurrence_id integer NULL,
    visit_detail_id integer NULL,
    note_source_value varchar(50) NULL,
    note_event_id bigint NULL,
    note_event_field_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.note_nlp (
    note_nlp_id integer NOT NULL,
    note_id integer NOT NULL,
    section_concept_id integer NULL,
    snippet varchar(250) NULL,
    lexical_variant varchar(250) NOT NULL,
    note_nlp_concept_id integer NULL,
    note_nlp_source_concept_id integer NULL,
    nlp_system varchar(250) NULL,
    nlp_date date NOT NULL,
    nlp_datetime TIMESTAMP NULL,
    term_exists varchar(1) NULL,
    term_temporal varchar(50) NULL,
    term_modifiers varchar(2000) NULL
);
""")

execute("""
CREATE TABLE omop.specimen (
    specimen_id integer NOT NULL,
    person_id integer NOT NULL,
    specimen_concept_id integer NOT NULL,
    specimen_type_concept_id integer NOT NULL,
    specimen_date date NOT NULL,
    specimen_datetime TIMESTAMP NULL,
    quantity NUMERIC NULL,
    unit_concept_id integer NULL,
    anatomic_site_concept_id integer NULL,
    disease_status_concept_id integer NULL,
    specimen_source_id varchar(50) NULL,
    specimen_source_value varchar(50) NULL,
    unit_source_value varchar(50) NULL,
    anatomic_site_source_value varchar(50) NULL,
    disease_status_source_value varchar(50) NULL
);
""")

execute("""
CREATE TABLE omop.fact_relationship (
    domain_concept_id_1 integer NOT NULL,
    fact_id_1 integer NOT NULL,
    domain_concept_id_2 integer NOT NULL,
    fact_id_2 integer NOT NULL,
    relationship_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.location (
    location_id integer NOT NULL,
    address_1 varchar(50) NULL,
    address_2 varchar(50) NULL,
    city varchar(50) NULL,
    state varchar(2) NULL,
    zip varchar(9) NULL,
    county varchar(20) NULL,
    location_source_value varchar(50) NULL,
    country_concept_id integer NULL,
    country_source_value varchar(80) NULL,
    latitude NUMERIC NULL,
    longitude NUMERIC NULL
);
""")

execute("""
CREATE TABLE omop.care_site (
    care_site_id integer NOT NULL,
    care_site_name varchar(255) NULL,
    place_of_service_concept_id integer NULL,
    location_id integer NULL,
    care_site_source_value varchar(50) NULL,
    place_of_service_source_value varchar(50) NULL
);
""")

execute("""
CREATE TABLE omop.provider (
    provider_id integer NOT NULL,
    provider_name varchar(255) NULL,
    npi varchar(20) NULL,
    dea varchar(20) NULL,
    specialty_concept_id integer NULL,
    care_site_id integer NULL,
    year_of_birth integer NULL,
    gender_concept_id integer NULL,
    provider_source_value varchar(50) NULL,
    specialty_source_value varchar(50) NULL,
    specialty_source_concept_id integer NULL,
    gender_source_value varchar(50) NULL,
    gender_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.payer_plan_period (
    payer_plan_period_id integer NOT NULL,
    person_id integer NOT NULL,
    payer_plan_period_start_date date NOT NULL,
    payer_plan_period_end_date date NOT NULL,
    payer_concept_id integer NULL,
    payer_source_value varchar(50) NULL,
    payer_source_concept_id integer NULL,
    plan_concept_id integer NULL,
    plan_source_value varchar(50) NULL,
    plan_source_concept_id integer NULL,
    sponsor_concept_id integer NULL,
    sponsor_source_value varchar(50) NULL,
    sponsor_source_concept_id integer NULL,
    family_source_value varchar(50) NULL,
    stop_reason_concept_id integer NULL,
    stop_reason_source_value varchar(50) NULL,
    stop_reason_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.cost (
    cost_id integer NOT NULL,
    cost_event_id integer NOT NULL,
    cost_domain_id varchar(20) NOT NULL,
    cost_type_concept_id integer NOT NULL,
    currency_concept_id integer NULL,
    total_charge NUMERIC NULL,
    total_cost NUMERIC NULL,
    total_paid NUMERIC NULL,
    paid_by_payer NUMERIC NULL,
    paid_by_patient NUMERIC NULL,
    paid_patient_copay NUMERIC NULL,
    paid_patient_coinsurance NUMERIC NULL,
    paid_patient_deductible NUMERIC NULL,
    paid_by_primary NUMERIC NULL,
    paid_ingredient_cost NUMERIC NULL,
    paid_dispensing_fee NUMERIC NULL,
    payer_plan_period_id integer NULL,
    amount_allowed NUMERIC NULL,
    revenue_code_concept_id integer NULL,
    revenue_code_source_value varchar(50) NULL,
    drg_concept_id integer NULL,
    drg_source_value varchar(3) NULL
);
""")

execute("""
CREATE TABLE omop.drug_era (
    drug_era_id integer NOT NULL,
    person_id integer NOT NULL,
    drug_concept_id integer NOT NULL,
    drug_era_start_date TIMESTAMP NOT NULL,
    drug_era_end_date TIMESTAMP NOT NULL,
    drug_exposure_count integer NULL,
    gap_days integer NULL
);
""")

execute("""
CREATE TABLE omop.dose_era (
    dose_era_id integer NOT NULL,
    person_id integer NOT NULL,
    drug_concept_id integer NOT NULL,
    unit_concept_id integer NOT NULL,
    dose_value NUMERIC NOT NULL,
    dose_era_start_date TIMESTAMP NOT NULL,
    dose_era_end_date TIMESTAMP NOT NULL
);
""")

execute("""
CREATE TABLE omop.condition_era (
    condition_era_id integer NOT NULL,
    person_id integer NOT NULL,
    condition_concept_id integer NOT NULL,
    condition_era_start_date TIMESTAMP NOT NULL,
    condition_era_end_date TIMESTAMP NOT NULL,
    condition_occurrence_count integer NULL
);
""")

execute("""
CREATE TABLE omop.episode (
    episode_id bigint NOT NULL,
    person_id bigint NOT NULL,
    episode_concept_id integer NOT NULL,
    episode_start_date date NOT NULL,
    episode_start_datetime TIMESTAMP NULL,
    episode_end_date date NULL,
    episode_end_datetime TIMESTAMP NULL,
    episode_parent_id bigint NULL,
    episode_number integer NULL,
    episode_object_concept_id integer NOT NULL,
    episode_type_concept_id integer NOT NULL,
    episode_source_value varchar(50) NULL,
    episode_source_concept_id integer NULL
);
""")

execute("""
CREATE TABLE omop.episode_event (
    episode_id bigint NOT NULL,
    event_id bigint NOT NULL,
    episode_event_field_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.metadata (
    metadata_id integer NOT NULL,
    metadata_concept_id integer NOT NULL,
    metadata_type_concept_id integer NOT NULL,
    name varchar(250) NOT NULL,
    value_as_string varchar(250) NULL,
    value_as_concept_id integer NULL,
    value_as_number NUMERIC NULL,
    metadata_date date NULL,
    metadata_datetime TIMESTAMP NULL
);
""")

execute("""
CREATE TABLE omop.cdm_source (
    cdm_source_name varchar(255) NOT NULL,
    cdm_source_abbreviation varchar(25) NOT NULL,
    cdm_holder varchar(255) NOT NULL,
    source_description TEXT NULL,
    source_documentation_reference varchar(255) NULL,
    cdm_etl_reference varchar(255) NULL,
    source_release_date date NOT NULL,
    cdm_release_date date NOT NULL,
    cdm_version varchar(10) NULL,
    cdm_version_concept_id integer NOT NULL,
    vocabulary_version varchar(20) NOT NULL
);
""")

execute("""
CREATE TABLE omop.concept (
    concept_id integer NOT NULL,
    concept_name varchar(255) NOT NULL,
    domain_id varchar(20) NOT NULL,
    vocabulary_id varchar(20) NOT NULL,
    concept_class_id varchar(20) NOT NULL,
    standard_concept varchar(1) NULL,
    concept_code varchar(50) NOT NULL,
    valid_start_date date NOT NULL,
    valid_end_date date NOT NULL,
    invalid_reason varchar(1) NULL
);
""")

execute("""
CREATE TABLE omop.vocabulary (
    vocabulary_id varchar(20) NOT NULL,
    vocabulary_name varchar(255) NOT NULL,
    vocabulary_reference varchar(255) NULL,
    vocabulary_version varchar(255) NULL,
    vocabulary_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.domain (
    domain_id varchar(20) NOT NULL,
    domain_name varchar(255) NOT NULL,
    domain_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.concept_class (
    concept_class_id varchar(20) NOT NULL,
    concept_class_name varchar(255) NOT NULL,
    concept_class_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.concept_relationship (
    concept_id_1 integer NOT NULL,
    concept_id_2 integer NOT NULL,
    relationship_id varchar(20) NOT NULL,
    valid_start_date date NOT NULL,
    valid_end_date date NOT NULL,
    invalid_reason varchar(1) NULL
);
""")

execute("""
CREATE TABLE omop.relationship (
    relationship_id varchar(20) NOT NULL,
    relationship_name varchar(255) NOT NULL,
    is_hierarchical varchar(1) NOT NULL,
    defines_ancestry varchar(1) NOT NULL,
    reverse_relationship_id varchar(20) NOT NULL,
    relationship_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.concept_synonym (
    concept_id integer NOT NULL,
    concept_synonym_name varchar(1000) NOT NULL,
    language_concept_id integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.concept_ancestor (
    ancestor_concept_id integer NOT NULL,
    descendant_concept_id integer NOT NULL,
    min_levels_of_separation integer NOT NULL,
    max_levels_of_separation integer NOT NULL
);
""")

execute("""
CREATE TABLE omop.source_to_concept_map (
    source_code varchar(50) NOT NULL,
    source_concept_id integer NOT NULL,
    source_vocabulary_id varchar(20) NOT NULL,
    source_code_description varchar(255) NULL,
    target_concept_id integer NOT NULL,
    target_vocabulary_id varchar(20) NOT NULL,
    valid_start_date date NOT NULL,
    valid_end_date date NOT NULL,
    invalid_reason varchar(1) NULL
);
""")

execute("""
CREATE TABLE omop.drug_strength (
    drug_concept_id integer NOT NULL,
    ingredient_concept_id integer NOT NULL,
    amount_value NUMERIC NULL,
    amount_unit_concept_id integer NULL,
    numerator_value NUMERIC NULL,
    numerator_unit_concept_id integer NULL,
    denominator_value NUMERIC NULL,
    denominator_unit_concept_id integer NULL,
    box_size integer NULL,
    valid_start_date date NOT NULL,
    valid_end_date date NOT NULL,
    invalid_reason varchar(1) NULL
);
""")

execute("""
CREATE TABLE omop.cohort (
    cohort_definition_id integer NOT NULL,
    subject_id integer NOT NULL,
    cohort_start_date date NOT NULL,
    cohort_end_date date NOT NULL
);
""")

execute("""
CREATE TABLE omop.cohort_definition (
    cohort_definition_id integer NOT NULL,
    cohort_definition_name varchar(255) NOT NULL,
    cohort_definition_description TEXT NULL,
    definition_type_concept_id integer NOT NULL,
    cohort_definition_syntax TEXT NULL,
    subject_concept_id integer NOT NULL,
    cohort_initiation_date date NULL
);
""")

print("All OMOP CDM tables created.")

## Import OMOP Vocabulary
Vocabulary CSVs are loaded with pandas. The `valid_start_date` / `valid_end_date` columns are stored as integers (YYYYMMDD) in Athena exports and are converted to proper dates before loading.

In [ ]:
def parse_omop_dates(df):
    for col in ('valid_start_date', 'valid_end_date'):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col].astype(str), format='%Y%m%d').dt.date
    return df

# CONCEPT
df_concept = parse_omop_dates(pd.read_csv('CONCEPT.csv'))
df_concept.to_sql('concept', engine, schema='omop', if_exists='append', index=False)
query("SELECT * FROM omop.concept LIMIT 5")

In [ ]:
# CONCEPT_RELATIONSHIP
df_cr = parse_omop_dates(pd.read_csv('CONCEPT_RELATIONSHIP.csv'))
df_cr.to_sql('concept_relationship', engine, schema='omop', if_exists='append', index=False)
query("SELECT * FROM omop.concept_relationship LIMIT 5")

In [ ]:
# CONCEPT_ANCESTOR
pd.read_csv('CONCEPT_ANCESTOR.csv').to_sql(
    'concept_ancestor', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.concept_ancestor LIMIT 5")

In [ ]:
# CONCEPT_SYNONYM
pd.read_csv('CONCEPT_SYNONYM.csv').to_sql(
    'concept_synonym', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.concept_synonym LIMIT 5")

In [ ]:
# VOCABULARY
pd.read_csv('VOCABULARY.csv').to_sql(
    'vocabulary', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.vocabulary LIMIT 5")

In [ ]:
# CONCEPT_CLASS
pd.read_csv('CONCEPT_CLASS.csv').to_sql(
    'concept_class', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.concept_class LIMIT 5")

In [ ]:
# DOMAIN
pd.read_csv('DOMAIN.csv').to_sql(
    'domain', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.domain LIMIT 5")

In [ ]:
# RELATIONSHIP
pd.read_csv('RELATIONSHIP.csv').to_sql(
    'relationship', engine, schema='omop', if_exists='append', index=False
)
query("SELECT * FROM omop.relationship LIMIT 5")

## Fill CDM Source

In [ ]:
execute("""
INSERT INTO omop.cdm_source (
    cdm_source_name, cdm_source_abbreviation, cdm_holder,
    source_release_date, cdm_release_date,
    cdm_version_concept_id, vocabulary_version
)
VALUES (
    'HEMA', 'HEMA', 'HEMA',
    '2026-05-06', '2026-05-06',
    756265,  -- CDM version 5.4
    'v5.0 27-FEB-25'
);
""")

query("SELECT * FROM omop.cdm_source")

## Import HEMA Source Data

In [ ]:
execute("CREATE SCHEMA IF NOT EXISTS import")

df_dictionary = pd.read_csv('synthetic_DataDictionary_2026-05-04.csv')
df_dictionary.to_sql('dictionary', engine, schema='import', if_exists='replace', index=False)

df_mappings = pd.read_csv('synthetic_DataDictionary_OMOPMappings_2026-05-04 (1).csv')
df_mappings.to_sql('mappings', engine, schema='import', if_exists='replace', index=False)

df_labels = pd.read_csv('synthetic_output_labels_balanced_transfusion_gaussianv3.csv',
                        parse_dates=["Patient's date of birth"])
df_labels.to_sql('labels', engine, schema='import', if_exists='replace', index=False)

print("Source data imported.")

In [ ]:
print("--- dictionary ---")
display(query("SELECT * FROM import.dictionary LIMIT 5"))

print("--- mappings ---")
display(query("SELECT * FROM import.mappings LIMIT 5"))

print("--- labels ---")
display(query("SELECT * FROM import.labels LIMIT 5"))

## ETL: Fill OMOP Tables
### PERSON
PostgreSQL uses `EXTRACT` instead of DuckDB's `year()` / `month()` / `day()` shorthand.

In [ ]:
execute("""
INSERT INTO omop.person (
    person_id, gender_concept_id,
    year_of_birth, month_of_birth, day_of_birth,
    race_concept_id, ethnicity_concept_id,
    person_source_value, gender_source_value
)
SELECT
    row_number() OVER () AS person_id,
    CASE "Patient's sex at birth"
        WHEN 'Male'   THEN 8507
        WHEN 'Female' THEN 8532
        ELSE 0
    END,
    EXTRACT(YEAR  FROM "Patient's date of birth")::integer,
    EXTRACT(MONTH FROM "Patient's date of birth")::integer,
    EXTRACT(DAY   FROM "Patient's date of birth")::integer,
    0,
    0,
    "Record ID",
    "Patient's sex at birth"
FROM import.labels;
""")

query("SELECT * FROM omop.person")

### OBSERVATION_PERIOD
No recording date available — all data is treated as measured on 2026-05-12.

In [ ]:
execute("""
INSERT INTO omop.observation_period (
    observation_period_id, person_id,
    observation_period_start_date, observation_period_end_date,
    period_type_concept_id
)
SELECT
    row_number() OVER (),
    person_id,
    '2026-05-12',
    '2026-05-12',
    32809  -- Case Report Form
FROM omop.person;
""")

query("SELECT * FROM omop.observation_period")

### VISIT_OCCURRENCE

In [ ]:
execute("""
INSERT INTO omop.visit_occurrence (
    visit_occurrence_id, person_id, visit_concept_id,
    visit_start_date, visit_start_datetime,
    visit_end_date, visit_end_datetime,
    visit_type_concept_id
)
SELECT
    row_number() OVER (),
    p.person_id,
    581476,  -- Home visit
    op.observation_period_start_date,
    op.observation_period_start_date,
    op.observation_period_end_date,
    op.observation_period_end_date,
    32809  -- Case report form
FROM omop.person p
JOIN omop.observation_period op ON op.person_id = p.person_id;
""")

query("SELECT * FROM omop.visit_occurrence")

### CONDITION_OCCURRENCE
Sequences generate continuous IDs across multiple inserts.

In [ ]:
execute("CREATE SEQUENCE condition_id_seq START 1;")

# Primary diagnosis
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT
    nextval('condition_id_seq'),
    p.person_id,
    CASE "Diagnosis retained by the specialised centre"
        WHEN 'Alpha-thalassemia'   THEN 4287844
        WHEN 'Beta-thalassemia'    THEN 65959000
        WHEN 'Sickle cell disease' THEN 25518  -- Sickle cell trait concept
        ELSE 0
    END,
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865,  -- Patient self report
    vo.visit_occurrence_id,
    "Diagnosis retained by the specialised centre"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

query("SELECT * FROM omop.condition_occurrence")

In [ ]:
# Hypopituitarism (no results expected — no 'Yes' values in synthetic data)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4254542,  -- Hypopituitarism
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Hypopituitarism"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Hypopituitarism" = 'Yes';
""")

# Infertile
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4311387,  -- Infertile
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Is the patient infertile?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient infertile?" = 'Yes';
""")

# Acute viral hepatitis (no results expected in synthetic data)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4211974,  -- Acute viral hepatitis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute viral hepatitis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute viral hepatitis" = 'Yes';
""")

# Vitamin D deficiency
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    436070,  -- Vitamin D deficiency
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Vitamin D deficiency"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Vitamin D deficiency" = 'Yes, confirmed';
""")

# Osteoporosis
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    80502,  -- Osteoporosis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteoporosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteoporosis" = 'Yes, confirmed';
""")

# Osteopenia
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4195039,  -- Osteopenia
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteopenia"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteopenia" = 'Yes, confirmed';
""")

# Acute chest syndrome
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    254062,  -- Acute chest syndrome
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute chest syndrome"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute chest syndrome" = 'Yes, confirmed';
""")

query("SELECT * FROM omop.condition_occurrence")

### OBSERVATION

In [ ]:
execute("CREATE SEQUENCE observation_id_seq START 1;")

# Country of birth
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    CASE l."Patient's country of birth"
        WHEN 'Australia'      THEN 4199969
        WHEN 'Canada'         THEN 4200105
        WHEN 'Congo'          THEN 4202085
        WHEN 'Cyprus'         THEN 4152209
        WHEN 'Greece'         THEN 4151604
        WHEN 'Iraq'           THEN 4152215
        WHEN 'Syria'          THEN 4153306
        WHEN 'United Kingdom' THEN 4202086  -- England
        ELSE 40482029  -- Country of birth unknown
    END,
    vo.visit_start_date, vo.visit_start_date,
    32865,  -- Patient self report
    vo.visit_occurrence_id,
    l."Patient's country of birth"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# Transfusion status
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    40758326,  -- Transfusion status qualitative
    vo.visit_start_date, vo.visit_start_date, 32865, vo.visit_occurrence_id,
    l."Does the patient require regular or occasional transfusions in the present (in the last 12 months) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBB Allele 1 (not represented in OHDSI vocabularies)
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,  -- Not represented in OHDSI Standardized Vocabularies
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 1", vo.visit_occurrence_id,
    'HGNC:4827'  -- From mappings file
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBB Allele 2
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 2", vo.visit_occurrence_id, 'HGNC:4827'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 1
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 1", vo.visit_occurrence_id, 'HGNC:4823;HGNC:4824'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 2
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 2", vo.visit_occurrence_id, 'HGNC:HGNC:4823;HGNC:4824'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# Drug compliance — poor
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4292063,  -- Drug compliance poor
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Poor (< 60% of recommended dose)';
""")

# Drug compliance — good / excellent
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4056965,  -- Drug compliance good
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Good (80-60 % of recommended dose)'
   OR l."Drug compliance" = 'Excellent (>80% of recommended dose)';
""")

query("SELECT * FROM omop.observation")

### MEASUREMENT

In [ ]:
execute("CREATE SEQUENCE measurement_id_seq START 1;")

# Cardiac iron T2* (ms) — no OMOP concept available
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- No suitable mapping
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Cardiac iron T2*  (milisec)",
    9593,  -- millisecond
    vo.visit_occurrence_id, 'Cardiac iron T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cardiac iron T2*  (milisec)" IS NOT NULL;
""")

# Liver MRI T2* (ms)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- No suitable mapping
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Liver MRI T2* (milisec)",
    9593, vo.visit_occurrence_id, 'Liver MRI T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Liver MRI T2* (milisec)" IS NOT NULL;
""")

# Cirrhosis
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, unit_concept_id,
    visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    36770062,  -- Cirrhosis
    vo.visit_start_date, vo.visit_start_date, 32809,
    9593, vo.visit_occurrence_id, l."Cirrhosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cirrhosis" = 'Yes, confirmed';
""")

# Ferritin serum (ng/mL)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    37208753,  -- Ferritin mass concentration in serum
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Ferritin serum (ng/mL / µg/L)",
    8842,  -- ng/mL
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Ferritin serum (ng/mL / µg/L)" IS NOT NULL;
""")

# Serum iron (μg/dL)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    4097596,  -- Serum iron measurement
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Serum iron (μg/dL)",
    8837,  -- ug/dL
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Serum iron (μg/dL)" IS NOT NULL;
""")

query("SELECT * FROM omop.measurement")

### PROCEDURE_OCCURRENCE

In [ ]:
execute("CREATE SEQUENCE procedure_id_seq START 1;")

# Chelation treatment
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    4068544,  -- Chelation (of cornea — verify mapping)
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on chelation treatment ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on chelation treatment ?" = 'Yes';
""")

# Hydroxyurea treatment
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    3169902,  -- Hydroxyurea therapy
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on hydroxyurea treatment (present time) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on hydroxyurea treatment (present time) ?" = 'Yes';
""")

# Splenectomy
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    2834904,  -- Removal of spleen (Lymphatic and Hemic Systems)
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Has the spleen been removed ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Has the spleen been removed ?" = 'Yes, totally';
""")

query("SELECT * FROM omop.procedure_occurrence")

## Final Overview

In [ ]:
query("""
SELECT table_schema, table_name,
       (xpath('/row/cnt/text()',
              query_to_xml(format('SELECT COUNT(*) AS cnt FROM %I.%I', table_schema, table_name),
                           false, true, '')))[1]::text::integer AS row_count
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, table_name
""")